### intro to this notebook

## Personally identifiable information (PII) in the dataset

###  Definition of PII

Personally Identifiable Information (PII) refers to any information that can identify a specific individual, either directly or indirectly.

Under the General Data Protection Regulation (GDPR), the broader term used is “personal data”, defined as "Any information relating to an identified or identifiable natural person."

An identifiable person is someone who can be identified directly (e.g., by name or ID number) or indirectly (e.g., through a combination of attributes such as date of birth, gender, and ZIP code).

Since the NovaCred dataset is used for credit scoring decisions, it contains multiple forms of personal data that fall under GDPR protection and EU AI Act obligations.


###  PII classification
We have audited the `raw_credit_applications.json` dataset and identified several categories of Personally Identifiable Information (PII). Under GDPR, these require specific technical and organizational measures to ensure lawful processing.

#### 1 Direct Identifiers
These fields can be used to uniquely identify a natural person without any additional information:
* `full_name`: Directly identifies a person. It's the most straightforward form of PII since it points to a specific individual without needing any additional context.
* `email`: Uniquely assigned to an individual, often contains the person's name, and serves as both an identifier and a contact channel. Even without a name attached, an email address can be traced back to its owner.
* `ssn`: A permanent, unique national identifier assigned to a single person. It's among the most sensitive PII because it cannot be easily changed and is widely used for identity verification across financial, governmental, and healthcare systems. Exposure creates severe risk of identity theft.
* `ip_adress`: Identifies the device and network used during the application. It can be linked back to a specific individual through their internet service provider, which means it qualifies as personal data even if it doesn't directly contain a name.






#### 2 Indirect identifiers (quasi-identifiers)

Not identifying on their own, but become PII when combined with other fields:

* `date_of_birth`:  A birthdate alone is shared by many people. However, when combined with other quasi-identifiers like gender and zip code, the pool of matching individuals shrinks dramatically, often to a single person. The risk lies in combination, not isolation.

* `gender`: Same principle. Knowing someone's gender tells you very little by itself, but it significantly narrows the population when crossed with location and age data. It's also a protected characteristic under anti-discrimination legislation, adding a layer of sensitivity beyond identification.

* `zip_code`: Geographic information that narrows someone's location to a small area. Some postal codes cover very few residents, which makes re-identification much easier. It also carries indirect sensitivity because residential patterns often correlate with ethnicity and socioeconomic status.

#### 3 Behavioral Data with Privacy Implications

* `spending_behavior` (category + amount): Not a traditional identifier, but highly personal. Detailed spending patterns can reveal sensitive aspects of someone's life such as health conditions, religious beliefs, or political affiliations depending on the categories tracked. This raises a data minimization concern: is this level of granularity necessary for the purpose of credit decisioning?

#### 4 Non-PII fields

* `_id`: An internal application identifier with no meaning outside the system. It doesn't relate to the person's real-world identity.

* `annual_income`, `credit_history_months`, `debt_to_income`, `savings_balance`: These are financial attributes, not identifiers. They describe characteristics rather than pointing to a specific person. However, they become personal data the moment they are linked to any of the identifiers above, because they then describe an identified individual's financial situation.

* decision object (`loan_approved`, `interest_rate`, `approved_amount`, `rejection_reason`): Not PII on its own, but once linked to an applicant it represents the outcome of automated profiling. This makes it relevant under GDPR provisions on automated decision-making, since the decision directly affects the individual's access to credit.

## Implementing Pseudonymization

### SSN Pseudonymization Demo

The SSN (Social Security Number) field is the most sensitive direct identifier in the dataset. It is a permanent, unique national identifier that cannot be changed, making its exposure a severe identity theft risk.

To protect this field while preserving analytical utility, we apply **SHA-256 hashing**, a one-way cryptographic function that converts each SSN into a fixed-length string of characters.

**Why hashing works for this use case:**
- **Irreversible:** the original SSN cannot be recovered from the hash
- **Deterministic:** the same SSN always produces the same hash, so we can still detect duplicate records without seeing the real values
- **Fixed output:** regardless of input, the hash is always 64 characters, removing any structural clues about the original

**GDPR note:** hashed data is still considered *pseudonymized* (not *anonymized*) under GDPR Article 4(5), because re-identification remains theoretically possible. This means GDPR still applies to the hashed dataset.

In [2]:
import pandas as pd
import hashlib

df = pd.read_csv("cleaned_credit_applications.csv")

# Show original SSNs
print("=== Before Pseudonymization ===")
print(df[["_id", "full_name", "ssn"]].head())

# Apply SHA-256 hashing
def hash_ssn(ssn):
    return hashlib.sha256(ssn.encode()).hexdigest()

df["ssn_hashed"] = df["ssn"].apply(hash_ssn)

# Show the result
print("\n=== After Pseudonymization ===")
print(df[["_id", "full_name", "ssn", "ssn_hashed"]].head())

# Drop the original SSN from the working dataset
df = df.drop(columns=["ssn"])

print("\n=== Final: Original SSN removed from dataset ===")
print(df[["_id", "full_name", "ssn_hashed"]].head())

=== Before Pseudonymization ===
       _id        full_name          ssn
0  app_200      Jerry Smith  596-64-4340
1  app_037   Brandon Walker  425-69-4784
2  app_215      Scott Moore  370-78-5178
3  app_024       Thomas Lee  194-35-1833
4  app_184  Brian Rodriguez  480-41-2475

=== After Pseudonymization ===
       _id        full_name          ssn  \
0  app_200      Jerry Smith  596-64-4340   
1  app_037   Brandon Walker  425-69-4784   
2  app_215      Scott Moore  370-78-5178   
3  app_024       Thomas Lee  194-35-1833   
4  app_184  Brian Rodriguez  480-41-2475   

                                          ssn_hashed  
0  2caf30528c21a10e1307b27f9dbbfc312f0c00d46b333e...  
1  2f7da45fefdcfb2c5b4f5b6f1465c22054c36e04fc77c1...  
2  db120edcee2366a48d6d77c2db8c64c5536b8dc3c3c524...  
3  c835719be02018987096d6e49529a24b1d7e7ab35c84b1...  
4  41c7de40dc49185886e6ecb37346ec9eabce16087b7508...  

=== Final: Original SSN removed from dataset ===
       _id        full_name                  